In [12]:
import sqlite3
import pandas as pd

In [13]:
import logging

# ==========================================
# Logger configuratie
# ==========================================
logging.basicConfig(
    level=logging.INFO, # Vangt INFO, WARNING, ERROR en CRITICAL op
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("etl_proces.log"), # Sla op in een bestand
        logging.StreamHandler()                # Print ook in de output van je Jupyter Notebook
    ]
)

In [14]:
# Connectie met het Source Data Model
sdm_conn = sqlite3.connect('SDM.db')
sdm_cursor = sdm_conn.cursor()

In [15]:
# Connecties met de bron-databases
aaa = sqlite3.connect('BikeToDrive_1_Accessoireverkoop.db')
bbb = sqlite3.connect('BikeToDrive_2_Fietsverkoop.db')
ccc = sqlite3.connect('BikeToDrive_3_Onderhoud.db')
ddd = sqlite3.connect('BikeToDrive_4_Accessoire_inkoop.db')
eee = sqlite3.connect('BikeToDrive_5_Fiets_Inkoop.db')
# etc

In [16]:
# Stap A: De "Reset-knop" (Tabellen leegmaken)
def reset_sdm():
    # Foreign keys tijdelijk uit tijdens het leegmaken,
    # anders krijgen we foutmeldingen bij tabellen die aan elkaar gekoppeld zijn
    sdm_cursor.execute("PRAGMA foreign_keys = OFF;")

    # Lijst van alle tabellen in SDM
    tabellen = [
        "Accessoire_Inkoop", "Accessoire_Inkoop_Accessoire", "Accessoire_Inkoop_Leverancier",
        "Accessoireverkoop_Accessoire_Verkoop", "Accessoireverkoop_Monteur", "Accessoireverkoop_Accessoire",
        "Accessoireverkoop_Leverancier", "Accessoireverkoop_Klant", "Accessoireverkoop_Filiaal",
        "Onderhoud", "Onderhoud_Monteur", "Onderhoud_Filiaal", "Onderhoud_Fiets", "Onderhoud_Fabrikant",
        "Fiets_Inkoop", "Fiets_Inkoop_Fiets", "Fiets_Inkoop_Fabrikant",
        "Fietsverkoop_Fiets_Verkoop", "Fietsverkoop_Fiets", "Fietsverkoop_Fabrikant",
        "Fietsverkoop_Monteur", "Fietsverkoop_Klant", "Fietsverkoop_Filiaal"
    ]

    # ull Refresh principe
    for tabel in tabellen:
        sdm_cursor.execute(f"DELETE FROM {tabel};")

    sdm_conn.commit()

    # Zet foreign keys weer aan voor de data-integriteit tijdens het inladen
    sdm_cursor.execute("PRAGMA foreign_keys = ON;")
    print("Reset compleet: Alle tabellen in het SDM zijn leeg.")

# Voer de reset uit
reset_sdm()

Reset compleet: Alle tabellen in het SDM zijn leeg.


In [17]:
# Serzat code

# Stap B: Data overzetten (Bulk Load)
# def laad_data_over(bron_conn, query, doel_tabel):
#     """
#     Functie om data uit een bron te halen en als bulk in het SDM te schieten.
#     """
#     # Lees data uit de bron met Pandas
#     df = pd.read_sql_query(query, bron_conn)
#
#     # Schrijf data naar het SDM.
#     # if_exists='append' zorgt dat de data in jouw zelfgemaakte tabellen wordt gestopt.
#     df.to_sql(doel_tabel, sdm_conn, if_exists='append', index=False)
#     print(f"✅ Data succesvol ingeladen in {doel_tabel} ({len(df)} rijen).")

In [18]:
# -- Voorbeeld: Inladen van Accessoire Inkoop --
# Let op: de tabelnamen in de query ('SELECT * FROM leverancier') moeten
# overeenkomen met hoe ze in de BRON database heten.

try:
    logging.info("Start inladen van Accessoire Inkoop data...")

    # 1. Leveranciers inladen
    logging.info("Bezig met: Leverancier -> Accessoire_Inkoop_Leverancier")
    laad_data_over(
        bron_inkoop_conn,
        "SELECT * FROM Leverancier",
        "Accessoire_Inkoop_Leverancier"
    )

    # 2. Accessoires inladen
    logging.info("Bezig met: Accessoire -> Accessoire_Inkoop_Accessoire")
    laad_data_over(
        bron_inkoop_conn,
        "SELECT * FROM Accessoire",
        "Accessoire_Inkoop_Accessoire"
    )

    # 3. Inkoop records inladen
    logging.info("Bezig met: Inkoop -> Accessoire_Inkoop")
    laad_data_over(
        bron_inkoop_conn,
        "SELECT * FROM Inkoop",
        "Accessoire_Inkoop"
    )

    logging.info("✅ Data uit Accessoire Inkoop is succesvol ingeladen.")

except Exception as e:
    # Gebruik logging.error in plaats van print om fouten goed te markeren
    logging.error(f"❌ Er is een fout opgetreden bij het inladen: {e}")

2026-03-05 15:38:36,045 - INFO - Start inladen van Accessoire Inkoop data...
2026-03-05 15:38:36,047 - INFO - Bezig met: Leverancier -> Accessoire_Inkoop_Leverancier
--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\Bayra\AppData\Local\Temp\ipykernel_21360\1725340060.py", line 10, in <module>
    laad_data_over(
    ^^^^^^^^^^^^^^
NameError: name 'laad_data_over' is not defined

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\Bayra\AppData\Local\Programs\Python\Python313\Lib\logging\__init__.py", line 1154, in emit
    stream.write(msg + self.terminator)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Bayra\AppData\Local\Programs\Python\Python313\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode characte

In [19]:
aaa.close()
bbb.close()
ccc.close()
ddd.close()
eee.close()
sdm_conn.close()